# 02 - Classification: will a hotel booking be cancelled?
**Target:** `is_canceled` (1 = cancelled). **Metric focus:** F1 and recall for the cancelled class (a missed cancellation costs an empty room), plus ROC-AUC.

Run top to bottom. Step 1 builds the clean dataset; step 2 trains/tunes/evaluates (about 8 min on one CPU).

In [ ]:
import sys, pandas as pd
from pathlib import Path
from IPython.display import Image, display
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import prepare_data, train_classification

## 1. Data preparation (rule-based, leak-free)
Dropped `reservation_status`, `reservation_status_date` (direct target leakage) and `assigned_room_type` (only known at check-in); removed duplicates, impossible rows and extreme `adr`; engineered `total_nights`, `total_guests`, `has_kids`, `arrival_weekday`, `adr_per_guest`, etc.

In [ ]:
clean, log = prepare_data.prepare()
clean.to_csv(prepare_data.OUT, index=False)
print(log)
clean.head()

In [ ]:
clean['is_canceled'].value_counts(normalize=True).round(3)

## 2. Train, tune, evaluate
Stratified 80/20 split (seed 42) **before** fitting. Scaling/encoding live inside the Pipeline. Models compared with 5-fold CV on the training set only; the test set is used once at the end.

In [ ]:
train_classification.main()

## 3. Results

In [ ]:
FIG = ROOT / 'reports'
print('Cross-validation (train only)'); display(pd.read_csv(FIG/'classification_cv_results.csv'))
print('Held-out test set');            display(pd.read_csv(FIG/'classification_test_results.csv'))

In [ ]:
display(Image(FIG/'clf_confusion_matrices.png')); display(Image(FIG/'clf_roc_curves.png'))

## 4. Interpretation and error analysis

In [ ]:
display(Image(FIG/'clf_feature_importance.png'))
display(pd.read_csv(FIG/'clf_error_by_segment.csv'))
display(pd.read_csv(FIG/'clf_error_by_deposit.csv'))

**Takeaways**
- All tuned models beat the majority-class baseline (72% accuracy, 0 recall) by a wide margin.
- Random Forest has the best F1 (CV and test); Gradient Boosting has the best accuracy/AUC and precision but misses more cancellations.
- Most errors are missed cancellations from `Online TA` bookings with no deposit - the hardest group.
- `country` and `agent` rank high partly because they act like identifiers for booking behaviour; treat with care.